# Enrich Contacts with Emails Using DropContact API

- This script reads contact data (first name, last name, company) from a Google Sheet,
- queries the DropContact API to retrieve email addresses and phone numbers,
- and writes the enriched data back to a new sheet in the same Google Sheet.
- It handles batching, API authentication, and phone number formatting for Google Sheets.


In [1]:
!sudo /bin/bash -c "(source /venv/bin/activate; pip install --upgrade google-api-python-client)"
!sudo /bin/bash -c "(source /venv/bin/activate; pip install gspread-pandas)"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.0/14.0 MB 36.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.8/160.8 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.1/216.1 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.9/96.9 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.5/294.5 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 321.1/321.1 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.3/181.3 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.1/83.1 kB 7.8 MB/s eta 0:00:00


# Import

In [2]:
import os

import gspread_pandas
import pandas as pd

import helpers.hgoogle_drive_api as hgofiapi
import ck_marketing.dropcontact as mrkdrop

# Get data from Google Sheet

In [4]:
!ls .

SorrTask606_Get_email_from_dropcontact.ipynb
SorrTask606_Get_email_from_dropcontact.py


In [13]:
creds = hgofiapi.get_credentials()

In [15]:
# Set up the Google sheet name.
# gsheet_name = "Search4.FinTech_VC_in_US.SalesNavigator"
gsheet_name = "https://docs.google.com/spreadsheets/d/1FqXXx7NfGfO8xDjbNlANqWJx0wmeXX95BaXy115W-9c/edit#gid=41878666"
#
spread = gspread_pandas.Spread(gsheet_name, creds=creds)
df = spread.sheet_to_df(index=None, sheet="Missing emails")[:10]
print(df.shape)
df.head()

(10, 73)


,error,baseUrl,timestamp,linkedinProfileUrl,email,linkedinProfile,description,headline,location,imgUrl,...,skill6,endorsement6,profileId,schoolUrl2,jobDescription2,schoolDescription,schoolDescription2,mail,phoneNumber,facebookUrl
0,,https://www.linkedin.com/sales/lead/ACwAAAC_H4...,2023-11-11T23:30:18.253Z,https://www.linkedin.com/in/jajordan13/,,https://www.linkedin.com/in/jajordan13/,"As an early stage investor, strategic advisor,...","Venture Partner, iGlobe Partners - Powering Ga...","Boston, Massachusetts, United States",https://media.licdn.com/dms/image/C4D03AQEtZK0...,...,Early Stage Ventures,,jajordan13,https://www.linkedin.com/company/17245/,Mentor in Residence and advisor at Techstars B...,,,,,
1,,https://www.linkedin.com/sales/lead/ACwAAADgYd...,2023-11-11T23:30:55.109Z,https://www.linkedin.com/in/greg-arrese/,,https://www.linkedin.com/in/greg-arrese/,Early stage tech investor based in La Jolla,"Founder, Investor @ Ride Ventures","San Diego, California, United States",https://media.licdn.com/dms/image/C4E03AQFTvEc...,...,Equities,26,greg-arrese,https://www.linkedin.com/company/14629/,CRCM Ventures invests in seed and early stage ...,Section D,,,,
2,,https://www.linkedin.com/sales/lead/ACwAAAAWPo...,2023-11-11T23:34:44.320Z,https://www.linkedin.com/in/ptagare/,,https://www.linkedin.com/in/ptagare/,I manage National Grid's (NYSE: NGG) Corporate...,"VC Investments to enable a safe, clean, afford...",San Francisco Bay Area,https://media.licdn.com/dms/image/D5603AQEQjBz...,...,Venture Capital,,ptagare,https://www.linkedin.com/company/4099/,,The Kauffman Fellows Program is designed to de...,,,,
3,,https://www.linkedin.com/sales/lead/ACwAAAK9Mb...,2023-11-11T23:35:48.923Z,https://www.linkedin.com/in/wale-ayeni-53648113/,,https://www.linkedin.com/in/wale-ayeni-53648113/,Wale Ayeni has over 18 years of global technol...,Emerging and Frontier Markets Venture Capital ...,Washington DC-Baltimore Area,https://media.licdn.com/dms/image/C5603AQEP2T5...,...,Product Management,7,wale-ayeni-53648113,https://www.linkedin.com/company/3638/,,,,,,
4,,https://www.linkedin.com/sales/lead/ACwAAAGJd9...,2023-11-12T05:31:34.595Z,https://www.linkedin.com/in/eman-zadeh-78b7768/,,https://www.linkedin.com/in/eman-zadeh-78b7768/,"Experienced Founder skilled in Sales, Product ...",Founder & Fintech/Blockchain Investor (Early &...,"Los Angeles, California, United States",https://media.licdn.com/dms/image/D5603AQEfDae...,...,Product Marketing,,eman-zadeh-78b7768,https://www.linkedin.com/company/4418/,"Allows companies to launch accounts, cards and...",,,,,


# Set up

In [24]:
# Batch size is how many data we send to the API per request.
# Batch endpoint can process up to 250 contacts with a single request.
# Default batch size is set to 50, for an 1 minute processing time.
# One contact data must be less than 10 kB.
#
# The API will cost 1 credit per data length.
batch_size = 50
# The column titles for first name, last name and company name in Given GSheet.
first_name_col = "firstName"
last_name_col = "lastName"
company_col = "company"
# API key of DropContact.
api_key = os.environ["API_KEY"]

# Get emails from DropContact

In [25]:
email_df = mrkdrop.get_email_from_dropcontact(
    df[first_name_col], df[last_name_col], df[company_col], api_key, batch_size
)

Processing batches:   0%|                                                     | 0/1 [00:00<?, ?it/s]

Starting query batch 0.
Batch 0: Query ID: iudbjfgdmtkbnxd.


Processing batches: 100%|#############################################| 1/1 [00:43<00:00, 43.14s/it]

Batch 0: Query finished. Credits left: 8091.
Batch 0 completed in 43.13 seconds.
Total processing time: 43.16 seconds.


In [26]:
email_df

,first name,last name,full name,email,phone,pronoun,job title
0,Jennifer,Jordan,Jennifer Jordan,,+65 6478 9716,Mrs,
1,Greg,Arrese,Greg Arrese,,,Mr,Founder Managing Member
2,Pradeep,Tagare,Pradeep Tagare,,,Mr,
3,Wale,Ayeni,Wale Ayeni,,,Mr,Managing Partner
4,Eman,Zadeh,Eman Zadeh,,,Mrs,Investor
5,Max,Michaels,Max Michaels,,,Mr,Fintech
6,Mohammed,Almeshekah,Mohammed Almeshekah,,,Mr,
7,Jonathan,M Padilla 彭庄炜,Jonathan M Padilla 彭庄炜,,,Mr,CEO & Co-Founder
8,Medina,Ruben de Jesus,Medina Ruben de Jesus,,,Mrs,Venture Partner
9,Bert,Navarrete,Bert Navarrete,,,Mr,


# Write email_df to the same Google Sheet

In [93]:
# Fix phone number format.
def prepare_phone_number_for_sheets(phone_number):
    if phone_number != "":
        pattern = r"^"
        replacement = "'"
        return re.sub(pattern, replacement, phone_number)
    else:
        return phone_number


email_df["phone"] = email_df["phone"].apply(prepare_phone_number_for_sheets)

email_df

,first name,last name,full name,email,phone,pronoun,job title
0,Johanan,Ottensooser,Johanan Ottensooser,johanan.ottensooser@p72.vc,,Mr,
1,Chase,Garbers,Chase Garbers,,,Mr,
2,Sonali,Sambhus,Sonali Sambhus,,,Mrs,Advisory Board Member
3,David,Benham,David Benham,david@mighty.net,,Mr,
4,Russell,Deakin,Russell Deakin,,,Mr,CIO & Managing Partner
5,Jeff,Bell,Jeff Bell,jbell@midoceanpartners.com,'+1 212-497-1407,Mr,
6,Jeff,Clavier,Jeff Clavier,jeff@uncorkcapital.com,'+1 650-688-1801,Mr,
7,Nat,Clarkson,Nat Clarkson,,,Mr,
8,Ollie,Howie,Ollie Howie,ollie@nextgenvp.com,'+20 160909,Mr,
9,Danish,M.,Danish M.,,'+64 7 477 8020,Mr,


In [94]:
def df_to_gsheet(gsheet_name: str, df: pd.DataFrame) -> None:
    # Write to the sheet.
    # Make sure the sheet "email"(sheet_name) exists in the Google Sheet.
    sheet_name = "email"
    spread2 = gspread_pandas.Spread(
        gsheet_name, sheet=sheet_name, create_sheet=True, creds=creds
    )
    spread2.df_to_sheet(df, index=False)


#
df_to_gsheet(gsheet_name, email_df)